# 04. Plony GUS (BDL)

Plony w dt/ha, tylko dla województw. Bez konta.

In [ ]:
import requests
import pandas as pd
import matplotlib.pyplot as plt

BDL = "https://bdl.stat.gov.pl/api/v1"
CROPS = {4332: "pszenica", 4380: "rzepak", 4344: "jęczmień", 4374: "buraki cukrowe"}

In [ ]:
# plony: wojewodztwa x lata, po jednej uprawie
rows = []
for variable_id, crop in CROPS.items():
    params = {"unit-level": 2, "year": list(range(2010, 2025)), "format": "json", "page-size": 100}
    for unit in requests.get(f"{BDL}/data/by-variable/{variable_id}", params=params).json()["results"]:
        for value in unit["values"]:
            rows.append({"teryt": unit["id"][2:4], "region": unit["name"].lower(), "crop": crop,
                         "year": int(value["year"]), "yield_dt_ha": value["val"]})
yields = pd.DataFrame(rows)
print(len(yields), "wierszy")
yields.head()

In [ ]:
yields.groupby("crop")["yield_dt_ha"].describe().round(1)

Brak uprawy GUS zapisuje jako **0**:

In [ ]:
yields[yields["yield_dt_ha"] == 0]

Rzepak 2012 (po zimie 2011/12) i pszenica 2018 (susza):

In [ ]:
yields.groupby(["year", "crop"])["yield_dt_ha"].mean().unstack().plot(subplots=True, layout=(2, 2), figsize=(12, 6), marker="o")
plt.tight_layout()
plt.show()